In [14]:
def parse_clinvar_xml(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    records = []
    for cv_record in root.iter('VariationArchive'):
        name = cv_record.get('VariationName', 'N/A')
        var_id = cv_record.get('VariationID', 'N/A')
        
        # Get all Description tags and take the last one (aggregate classification)
        descriptions = [e.text for e in cv_record.iter('Description') if e.text]
        significance = descriptions[-1] if descriptions else 'Unknown'
        
        records.append({
            'variation_id': var_id,
            'name': name,
            'clinical_significance': significance,
        })
    return pd.DataFrame(records)

In [15]:
pcsk9_variants = parse_clinvar_xml('../data/raw/clinvar_pcsk9.xml')
ldlr_variants  = parse_clinvar_xml('../data/raw/clinvar_ldlr.xml')
apob_variants  = parse_clinvar_xml('../data/raw/clinvar_apob.xml')

print(f'PCSK9 total: {len(pcsk9_variants)}')
print(f'LDLR total:  {len(ldlr_variants)}')
print(f'APOB total:  {len(apob_variants)}')

PCSK9 total: 500
LDLR total:  500
APOB total:  500


In [16]:
def filter_pathogenic(df):
    mask = df['clinical_significance'].str.contains(
        'Pathogenic|Likely pathogenic', case=False, na=False
    ) & ~df['clinical_significance'].str.contains(
        'Conflicting', case=False, na=False
    )
    return df[mask].reset_index(drop=True)

In [17]:
pcsk9_path = filter_pathogenic(pcsk9_variants)
ldlr_path  = filter_pathogenic(ldlr_variants)
apob_path  = filter_pathogenic(apob_variants)

print(f'Pathogenic PCSK9: {len(pcsk9_path)}')
print(f'Pathogenic LDLR:  {len(ldlr_path)}')
print(f'Pathogenic APOB:  {len(apob_path)}')

Pathogenic PCSK9: 5
Pathogenic LDLR:  198
Pathogenic APOB:  37


In [19]:
pcsk9_path.to_csv('../data/processed/pcsk9_pathogenic_variants.csv', index=False)
ldlr_path.to_csv('../data/processed/ldlr_pathogenic_variants.csv', index=False)
apob_path.to_csv('../data/processed/apob_pathogenic_variants.csv', index=False)

print('All pathogenic variant tables saved.')

All pathogenic variant tables saved.


In [20]:
git add .
git commit -m "Fixed ClinVar parser, extracted pathogenic variants for PCSK9, LDLR, APOB"
git push origin main

SyntaxError: invalid syntax (2369209968.py, line 1)

In [18]:
import subprocess

input_fasta  = '../data/processed/pcsk9_variants_for_alignment.fasta'
output_aln   = '../data/processed/pcsk9_alignment.fasta'

result = subprocess.run(
    ['muscle', '-align', input_fasta, '-output', output_aln],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('Alignment complete. Output saved.')
else:
    print('MUSCLE error:', result.stderr)

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
from Bio import AlignIO

alignment = AlignIO.read(output_aln, 'fasta')
print(f'Aligned {len(alignment)} sequences, each {alignment.get_alignment_length()} positions')
for record in alignment:
    print(f'{record.id[:20]:<20} {str(record.seq[:80])}')

In [ ]:
import requests, json

url = 'https://rest.uniprot.org/uniprotkb/Q8NBP7?format=json'
r = requests.get(url)
uniprot_data = r.json()

# Extract domain feature annotations
features = uniprot_data.get('features', [])
domains = [f for f in features if f['type'] in ['Domain', 'Region', 'Active site', 'Binding site']]

domain_df = pd.DataFrame([{
    'type': d['type'],
    'description': d.get('description', 'N/A'),
    'start': d['location']['start']['value'],
    'end': d['location']['end']['value']
} for d in domains])

domain_df.to_csv('../data/processed/pcsk9_domains.csv', index=False)
print(domain_df)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

fig, ax = plt.subplots(figsize=(14, 4))

# Draw domain bars
colors = {'Domain': '#1D9E75', 'Region': '#534AB7', 'Active site': '#BA7517', 'Binding site': '#993C1D'}
for _, row in domain_df.iterrows():
    color = colors.get(row['type'], '#888780')
    ax.barh(0.5, row['end'] - row['start'], left=row['start'], height=0.3,
            color=color, alpha=0.7, label=row['description'])
    ax.text((row['start'] + row['end']) / 2, 0.5, row['description'][:12],
            ha='center', va='center', fontsize=7, color='white', fontweight='bold')

# Plot variant positions from your PCSK9 pathogenic variant table
def extract_position(name):
    match = re.search(r'p\.[A-Za-z]+([0-9]+)', name)
    return int(match.group(1)) if match else None

pcsk9_path['position_aa'] = pcsk9_path['name'].apply(extract_position)
positions = pcsk9_path['position_aa'].dropna()

ax.scatter(positions, [0.85] * len(positions), color='#E24B4A', s=30, zorder=5, alpha=0.7)

ax.set_xlabel('Amino acid position')
ax.set_title('PCSK9 pathogenic variant positions relative to domain structure', pad=10)
ax.set_yticks([])
ax.set_xlim(0, 700)
plt.tight_layout()
plt.savefig('../results/pcsk9_variant_domain_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [11]:
import xml.etree.ElementTree as ET

tree = ET.parse('../data/raw/clinvar_pcsk9.xml')
root = tree.getroot()

# Look at the first VariationArchive in detail
first = root[0]
print('VariationName:', first.get('VariationName'))
print()

# Print all tags nested inside to find where clinical significance lives
for elem in first.iter():
    if 'significance' in elem.tag.lower() or 'interpretation' in elem.tag.lower() or 'description' in elem.tag.lower():
        print(f'Tag: {elem.tag}, Text: {elem.text}, Attrib: {elem.attrib}')

VariationName: NM_174936.4(PCSK9):c.1518G>T (p.Lys506Asn)

Tag: Description, Text: Uncertain significance, Attrib: {'DateLastEvaluated': '2024-04-29', 'SubmissionCount': '1'}
Tag: Description, Text: Uncertain significance, Attrib: {}
Tag: SubmitterDescription, Text: None, Attrib: {'OrgID': '508197', 'SubmitterName': 'Shariant Australia, Australian Genomics', 'Type': 'behalf', 'OrganizationCategory': 'consortium'}


In [12]:
import xml.etree.ElementTree as ET

tree = ET.parse('../data/raw/clinvar_pcsk9.xml')
root = tree.getroot()

# Find all Description texts and count them
from collections import Counter
descriptions = []
for elem in root.iter('Description'):
    if elem.text:
        descriptions.append(elem.text)

counts = Counter(descriptions)
for significance, count in counts.most_common():
    print(f'{significance}: {count}')

Uncertain significance: 729
Likely benign: 396
Conflicting classifications of pathogenicity: 20
Likely pathogenic: 6
Benign: 4
Pathogenic: 4
Benign/Likely benign: 1
